# Full Experiment: Rule Pipeline + Targeted VLM Error Recheck

Notebook này chạy kịch bản thực nghiệm chính cho đề tài: đánh giá `rule_full` trên toàn bộ dataset, sau đó chỉ dùng VLM để kiểm tra lại các ảnh mà `rule_full` dự đoán sai. Các độ đo chính gồm `F1-macro`, `Recall-macro`, confusion matrix và error analysis.


## 1. Setup

Notebook đọc ảnh từ `dataset/image`, suy ra nhãn từ folder `violate` và `not_violate`. VLM không chạy trên toàn bộ dataset; VLM chỉ chạy trên error cases của `rule_full` để tiết kiệm thời gian và phục vụ error analysis.


In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import json
import re
import sys
import time
import traceback

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'main.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

DATASET_DIR = PROJECT_ROOT / 'dataset' / 'image'
SCENE_CONFIG = PROJECT_ROOT / 'configs' / 'scene_config.example.json'
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
EVAL_DIR = PROJECT_ROOT / 'outputs' / 'full_evaluation' / RUN_ID
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Evaluation dir:', EVAL_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from IPython.display import display
from PIL import Image

## 2. Experiment Settings

In [ ]:
MAX_IMAGES = None
TASK_FILTER = None

# VLM chỉ chạy trên các ảnh rule_full dự đoán sai.
RUN_TARGETED_VLM_RECHECK = True
MAX_VLM_ERROR_CASES = None
QWEN_MODEL = 'Qwen/Qwen2.5-VL-3B-Instruct'
VLM_MAX_NEW_TOKENS = 256

BASE_THRESHOLD = 0.25
BASE_TEXT_THRESHOLD = 0.20

LABEL_NAMES = ['not_violate', 'violate']

print({
    'max_images': MAX_IMAGES,
    'task_filter': TASK_FILTER,
    'run_targeted_vlm_recheck': RUN_TARGETED_VLM_RECHECK,
    'max_vlm_error_cases': MAX_VLM_ERROR_CASES,
    'qwen_model': QWEN_MODEL,
})


## 3. Load Dataset

Nhãn nhị phân được suy ra từ cấu trúc thư mục: `violate` tương ứng 1, `not_violate` tương ứng 0.

In [ ]:
def infer_label(image_path: Path) -> int:
    parts = set(image_path.relative_to(DATASET_DIR).parts)
    if 'not_violate' in parts:
        return 0
    if 'violate' in parts:
        return 1
    return -1

def infer_task(image_path: Path) -> str:
    parts = image_path.relative_to(DATASET_DIR).parts
    return parts[0] if parts else 'unknown'

image_paths = sorted(
    p for p in DATASET_DIR.rglob('*')
    if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}
)

records = []
for path in image_paths:
    y = infer_label(path)
    task = infer_task(path)
    if y < 0:
        continue
    if TASK_FILTER and task != TASK_FILTER:
        continue
    records.append({
        'image_path': path,
        'relative_path': str(path.relative_to(PROJECT_ROOT)),
        'task': task,
        'y_true': y,
        'label_name': LABEL_NAMES[y],
    })

if MAX_IMAGES is not None:
    records = records[:MAX_IMAGES]

df_dataset = pd.DataFrame([{k: v for k, v in r.items() if k != 'image_path'} for r in records])
display(df_dataset.groupby(['task', 'label_name']).size().reset_index(name='count'))
print('Total images:', len(records))

## 4. Define Experiment Scenarios

Notebook có hai dòng kết quả: `rule_full` là kết quả rule-based trên toàn bộ dataset; `rule_full_targeted_vlm_recheck` là kết quả sau khi chỉ dùng VLM kiểm tra lại các error cases của `rule_full`. Lưu ý: kịch bản targeted dùng ground-truth để chọn error cases, nên phù hợp cho error analysis/diagnostic hơn là mô phỏng triển khai thực tế.


In [ ]:
EXPERIMENT_SCENARIOS = [
    {
        'name': 'rule_full',
        'description': 'Detector + full scene config + rule engine, no VLM',
    },
    {
        'name': 'rule_full_targeted_vlm_recheck',
        'description': 'Rule_full predictions corrected by running VLM only on rule_full error cases',
    },
]

display(pd.DataFrame(EXPERIMENT_SCENARIOS))


## 5. Prediction Helpers

`rule_prediction` chuyển output nhiều violation candidates thành nhãn nhị phân. VLM recheck chỉ dùng cho các ảnh mà `rule_full` đã dự đoán sai.


In [ ]:
NO_VIOLATION = 'no_violation_or_insufficient_evidence'

def rule_prediction(result: dict):
    candidates = [v for v in result.get('violations', []) if v.get('type') != NO_VIOLATION]
    if not candidates:
        return 0, NO_VIOLATION, 0.0
    best = max(candidates, key=lambda x: x.get('confidence', 0.0))
    return 1, best.get('type', 'unknown'), float(best.get('confidence', 0.0))

def build_vlm_recheck_prompt(result: dict, rule_label: int, rule_type: str):
    payload = {
        'image_path': result.get('image_path'),
        'image_size': result.get('image_size'),
        'detections': result.get('detections', []),
        'facts': result.get('facts', {}),
        'rule_engine_candidates': result.get('violations', []),
        'rule_prediction': {
            'label': LABEL_NAMES[rule_label],
            'violation_type': rule_type,
        },
    }
    return f"""
You are checking a Vietnamese traffic violation from one image.
Your job is to verify the rule-engine prediction using the image.
Focus on visible traffic signs, red/yellow/green lights, lane markings, stop lines, and the target vehicle position.
If evidence is insufficient in a single image, return not_violate or uncertain rather than over-claiming.

Return strict JSON only:
{{
  "final_label": "violate | not_violate | uncertain",
  "final_violation_type": "illegal_parking | wrong_lane | crossing_red_light | no_violation_or_insufficient_evidence | uncertain",
  "confidence": 0.0,
  "reason": "short explanation"
}}

Detector and rule output:
{json.dumps(payload, ensure_ascii=False, indent=2)}
""".strip()

def parse_vlm_json(text: str):
    cleaned = text.strip()
    cleaned = re.sub(r'^```(?:json)?', '', cleaned).strip()
    cleaned = re.sub(r'```$', '', cleaned).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', cleaned, flags=re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise

def apply_vlm_recheck(vlm_reasoner, image_path: Path, result: dict, rule_label: int, rule_type: str, rule_score: float):
    prompt = build_vlm_recheck_prompt(result, rule_label, rule_type)
    raw_text = vlm_reasoner.reason(image_path=str(image_path), prompt=prompt)
    parsed = parse_vlm_json(raw_text)
    final_label = str(parsed.get('final_label', 'uncertain')).strip().lower()
    if final_label == 'violate':
        return 1, parsed.get('final_violation_type', rule_type), float(parsed.get('confidence', rule_score)), parsed.get('reason', ''), raw_text
    if final_label == 'not_violate':
        return 0, NO_VIOLATION, float(parsed.get('confidence', 0.5)), parsed.get('reason', ''), raw_text
    return rule_label, rule_type, rule_score, parsed.get('reason', ''), raw_text

## 6. Run Rule Pipeline, Then Targeted VLM Recheck

Bước 1 chạy `rule_full` trên toàn bộ dataset. Bước 2 lấy các error cases của `rule_full`, chạy Qwen VLM trên đúng các ảnh đó, rồi tạo kết quả corrected để so sánh.


In [ ]:
from illegal_detector.traffic_violation_pipeline import TrafficViolationPipeline

pipeline = TrafficViolationPipeline(
    threshold=BASE_THRESHOLD,
    text_threshold=BASE_TEXT_THRESHOLD,
)

prediction_rows = []
result_cache = {}

print('\n=== rule_full ===')
for index, item in enumerate(records, start=1):
    start_time = time.perf_counter()
    row = {
        'model': 'rule_full',
        'description': 'Detector + full scene config + rule engine, no VLM',
        'image_path': item['relative_path'],
        'task': item['task'],
        'y_true': item['y_true'],
        'status': 'ok',
        'error': '',
        'vlm_status': 'not_run',
    }
    try:
        result = pipeline.analyze(
            image_path=str(item['image_path']),
            scene_config_path=str(SCENE_CONFIG),
            annotate_path=None,
            segment_dir=None,
            run_vlm=False,
        )
        result_cache[item['relative_path']] = result
        y_pred, pred_type, score = rule_prediction(result)
        row.update({
            'y_pred': y_pred,
            'score': score,
            'predicted_type': pred_type,
            'num_detections': len(result.get('detections', [])),
            'num_violations': len(result.get('violations', [])),
            'vlm_reason': '',
            'vlm_raw': '',
            'rechecked_by_vlm': False,
        })
    except Exception as exc:
        row.update({
            'y_pred': -1,
            'score': 0.0,
            'predicted_type': 'error',
            'num_detections': 0,
            'num_violations': 0,
            'vlm_reason': '',
            'vlm_raw': '',
            'rechecked_by_vlm': False,
            'status': 'error',
            'error': ''.join(traceback.format_exception_only(type(exc), exc)).strip(),
        })

    row['predict_time_sec'] = time.perf_counter() - start_time
    prediction_rows.append(row)
    print(
        f"{index:03d}/{len(records):03d} {item['relative_path']} "
        f"true={LABEL_NAMES[item['y_true']]} pred={row['y_pred']} status={row['status']}"
    )
    if row['status'] == 'error':
        print('  ', row['error'])

rule_df = pd.DataFrame(prediction_rows)
rule_errors = rule_df[
    (rule_df['model'] == 'rule_full')
    & (rule_df['status'] == 'ok')
    & (rule_df['y_true'] != rule_df['y_pred'])
].copy()

if MAX_VLM_ERROR_CASES is not None:
    rule_errors = rule_errors.head(MAX_VLM_ERROR_CASES)

error_image_set = set(rule_errors['image_path'])
print('\nRule_full error cases selected for VLM:', len(error_image_set))

corrected_rows = []
vlm_reasoner = None
if RUN_TARGETED_VLM_RECHECK and len(error_image_set) > 0:
    print('\n=== targeted VLM recheck on rule_full errors ===')
    from illegal_detector.vlm_reasoner import QwenVLReasoner
    vlm_reasoner = QwenVLReasoner(model_id=QWEN_MODEL, max_new_tokens=VLM_MAX_NEW_TOKENS)

for _, base_row in rule_df[rule_df['model'] == 'rule_full'].iterrows():
    corrected = base_row.to_dict()
    corrected['model'] = 'rule_full_targeted_vlm_recheck'
    corrected['description'] = 'Rule_full with VLM recheck only on rule_full error cases'
    corrected['rechecked_by_vlm'] = False
    corrected['vlm_status'] = 'not_selected'

    should_recheck = (
        RUN_TARGETED_VLM_RECHECK
        and base_row['status'] == 'ok'
        and base_row['image_path'] in error_image_set
    )

    if should_recheck:
        item = next(r for r in records if r['relative_path'] == base_row['image_path'])
        result = result_cache.get(base_row['image_path'])
        start_time = time.perf_counter()
        corrected['rechecked_by_vlm'] = True
        try:
            y_pred, pred_type, score, vlm_reason, vlm_raw = apply_vlm_recheck(
                vlm_reasoner,
                item['image_path'],
                result,
                int(base_row['y_pred']),
                base_row['predicted_type'],
                float(base_row['score']),
            )
            corrected.update({
                'y_pred': y_pred,
                'score': score,
                'predicted_type': pred_type,
                'vlm_reason': vlm_reason,
                'vlm_raw': vlm_raw,
                'vlm_status': 'ok',
                'status': 'ok',
                'error': '',
                'predict_time_sec': float(base_row['predict_time_sec']) + (time.perf_counter() - start_time),
            })
        except Exception as exc:
            # Keep the original rule prediction so the corrected scenario still covers the same image set.
            corrected.update({
                'vlm_status': 'error',
                'vlm_reason': '',
                'vlm_raw': '',
                'status': 'ok',
                'error': 'VLM recheck failed; kept rule prediction. ' + ''.join(traceback.format_exception_only(type(exc), exc)).strip(),
                'predict_time_sec': float(base_row['predict_time_sec']) + (time.perf_counter() - start_time),
            })
        print(
            f"VLM {base_row['image_path']} "
            f"rule_pred={base_row['y_pred']} corrected={corrected['y_pred']} vlm_status={corrected['vlm_status']}"
        )

    corrected_rows.append(corrected)

prediction_rows.extend(corrected_rows)
pred_df = pd.DataFrame(prediction_rows)
pred_path = EVAL_DIR / 'predictions.csv'
pred_df.to_csv(pred_path, index=False)
print('Saved:', pred_path)
display(pred_df.head())


## 7. Main Metrics: F1-macro And Recall-macro

Bảng chính so sánh `rule_full` với kết quả sau targeted VLM recheck. Vì dataset mất cân bằng, `F1-macro` và `Recall-macro` quan trọng hơn accuracy.


In [ ]:
def compute_main_metrics(group: pd.DataFrame):
    ok = group[group['status'] == 'ok'].copy()
    y_true = ok['y_true'].astype(int)
    y_pred = ok['y_pred'].astype(int)
    return {
        'n': len(ok),
        'errors': int((group['status'] != 'ok').sum()),
        'accuracy': accuracy_score(y_true, y_pred) if len(ok) else 0.0,
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0) if len(ok) else 0.0,
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0) if len(ok) else 0.0,
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0) if len(ok) else 0.0,
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0) if len(ok) else 0.0,
        'mean_predict_time_sec': ok['predict_time_sec'].mean() if len(ok) else 0.0,
        'vlm_rechecked': int(ok.get('rechecked_by_vlm', pd.Series(dtype=bool)).sum()) if len(ok) else 0,
        'vlm_errors': int((ok.get('vlm_status', pd.Series(dtype=str)) == 'error').sum()) if len(ok) else 0,
    }

metrics = []
for model, group in pred_df.groupby('model'):
    row = {'model': model}
    row.update(compute_main_metrics(group))
    metrics.append(row)

metrics_df = pd.DataFrame(metrics).sort_values('f1_macro', ascending=False)
metrics_path = EVAL_DIR / 'main_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)
display(metrics_df)
print('Saved:', metrics_path)

In [ ]:
plot_df = metrics_df.set_index('model')[['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']]
ax = plot_df.plot(kind='bar', figsize=(11, 5), ylim=(0, 1), title='Main metrics by experiment config')
ax.set_ylabel('Score')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(EVAL_DIR / 'main_metrics_bar.png', dpi=200)
plt.show()

## 8. Confusion Matrix

Confusion matrix được vẽ cho từng cấu hình để thấy false positive và false negative.

In [ ]:
confusion_rows = []

for model, group in pred_df[pred_df['status'] == 'ok'].groupby('model'):
    y_true = group['y_true'].astype(int)
    y_pred = group['y_pred'].astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    confusion_rows.append({
        'model': model,
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
    })

    disp = ConfusionMatrixDisplay(cm, display_labels=LABEL_NAMES)
    disp.plot(cmap='Blues', values_format='d')
    plt.title(f'Confusion matrix - {model}')
    plt.tight_layout()
    plt.savefig(EVAL_DIR / f'confusion_matrix_{model}.png', dpi=200)
    plt.show()

confusion_df = pd.DataFrame(confusion_rows)
confusion_df.to_csv(EVAL_DIR / 'confusion_matrices.csv', index=False)
display(confusion_df)

## 9. Error Analysis

Phần này ưu tiên phân tích lỗi của `rule_full`, sau đó xem VLM targeted recheck có sửa được lỗi nào hay không.


In [ ]:
rule_errors_for_analysis = pred_df[
    (pred_df['model'] == 'rule_full')
    & (pred_df['status'] == 'ok')
    & (pred_df['y_true'] != pred_df['y_pred'])
].copy()

corrected_df = pred_df[pred_df['model'] == 'rule_full_targeted_vlm_recheck'].copy()
error_compare = rule_errors_for_analysis.merge(
    corrected_df[['image_path', 'y_pred', 'predicted_type', 'score', 'vlm_reason', 'vlm_status', 'rechecked_by_vlm', 'status']],
    on='image_path',
    how='left',
    suffixes=('_rule', '_vlm_corrected'),
)
error_compare['vlm_fixed'] = error_compare['y_true'] == error_compare['y_pred_vlm_corrected']

error_path = EVAL_DIR / 'rule_full_error_analysis.csv'
error_compare.to_csv(error_path, index=False)

print('Rule_full error cases:', len(rule_errors_for_analysis))
if len(error_compare):
    print('VLM fixed cases:', int(error_compare['vlm_fixed'].sum()))
    print('VLM failed cases:', int((error_compare['vlm_status_vlm_corrected'] == 'error').sum()))
    display(error_compare[[
        'image_path', 'task', 'y_true', 'y_pred_rule', 'predicted_type_rule',
        'y_pred_vlm_corrected', 'predicted_type_vlm_corrected', 'vlm_status_vlm_corrected',
        'vlm_fixed', 'vlm_reason_vlm_corrected'
    ]].head(20))
else:
    display(error_compare)
print('Saved:', error_path)


In [ ]:
for _, row in error_compare.head(8).iterrows():
    corrected_label = 'N/A'
    if pd.notna(row.get('y_pred_vlm_corrected')):
        corrected_label = LABEL_NAMES[int(row['y_pred_vlm_corrected'])]

    print('\nImage:', row['image_path'])
    print(
        'True:', LABEL_NAMES[int(row['y_true'])],
        '| Rule pred:', LABEL_NAMES[int(row['y_pred_rule'])],
        '| VLM-corrected pred:', corrected_label,
        '| VLM status:', row.get('vlm_status_vlm_corrected', 'N/A'),
        '| Fixed:', row.get('vlm_fixed', False),
    )
    print('Rule type:', row['predicted_type_rule'])
    if row.get('vlm_reason_vlm_corrected'):
        print('VLM reason:', row['vlm_reason_vlm_corrected'])
    print('Analysis guide: kiểm tra detector có nhầm object không, scene polygon có quá rộng không, biển báo/đèn tín hiệu có đọc được không, ảnh đơn có thiếu bằng chứng thời gian không.')
    display(Image.open(PROJECT_ROOT / row['image_path']))


## 10. Report-Ready Summary

Cell này sinh đoạn tóm tắt ngắn có thể đưa vào báo cáo.

In [ ]:
best = metrics_df.iloc[0]
rule_row = metrics_df[metrics_df['model'] == 'rule_full'].iloc[0]
summary = (
    f"Rule_full achieved F1-macro={rule_row['f1_macro']:.3f} and Recall-macro={rule_row['recall_macro']:.3f}. "
    f"The best reported configuration is {best['model']}, with F1-macro={best['f1_macro']:.3f} "
    f"and Recall-macro={best['recall_macro']:.3f}. "
    f"Average prediction time for the best configuration is {best['mean_predict_time_sec']:.2f} seconds per image. "
    "The targeted VLM recheck is an error-analysis setting: it runs VLM only on rule_full error cases rather than on every image."
)
print(summary)
(EVAL_DIR / 'report_summary.txt').write_text(summary, encoding='utf-8')
